In [1]:
!pip install -q transformers torch torchvision

from transformers import pipeline

from PIL import Image

import numpy as np

import torch

device = 0 if torch.cuda.is_available() else -1

print(f'Используем: {"GPU" if device == 0 else "CPU"}')

Используем: GPU


## Загрузка всего и вся

In [2]:
!gdown --id 1kkK-qZO-5hwyybCFuaEvANlRXiv-5xxj

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1kkK-qZO-5hwyybCFuaEvANlRXiv-5xxj
From (redirected): https://drive.google.com/uc?id=1kkK-qZO-5hwyybCFuaEvANlRXiv-5xxj&confirm=t&uuid=d7877b15-d810-405b-ac29-57f2f1dd08c2
To: /content/vkitti_2.0.3_depth.tar
100% 8.15G/8.15G [01:57<00:00, 69.1MB/s]


In [3]:
!gdown 1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS

Downloading...
From (original): https://drive.google.com/uc?id=1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS
From (redirected): https://drive.google.com/uc?id=1Ib0Kk8xFoiexv6_9DRnSI58sTQHIDjjS&confirm=t&uuid=b6bcc865-445c-4f36-bc58-3fb3585c0d89
To: /content/vkitti_2.0.3_rgb.tar
100% 7.53G/7.53G [01:36<00:00, 78.4MB/s]


In [4]:
import os, tarfile


VKITTI_LOCAL = '/content/vkitti2'

os.makedirs(VKITTI_LOCAL, exist_ok=True)


for archive in ['vkitti_2.0.3_rgb.tar', 'vkitti_2.0.3_depth.tar']:

    archive_path = f'/content/{archive}'

    if os.path.exists(archive_path):


        print(f'📦 Распаковываю {archive}...')

        with tarfile.open(archive_path, 'r') as tar:

            tar.extractall(VKITTI_LOCAL)

        print(f'   ✅ Готово')

    else:

        print(f'   ❌ Не найден: {archive}')

📦 Распаковываю vkitti_2.0.3_rgb.tar...


/tmp/ipykernel_12443/3399381077.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(VKITTI_LOCAL)


   ✅ Готово
📦 Распаковываю vkitti_2.0.3_depth.tar...
   ✅ Готово


In [5]:
!gdown --id  1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8
From (redirected): https://drive.google.com/uc?id=1Dq6kJANt1yvpo3DOqhW-77abl-Gvi5v8&confirm=t&uuid=7f9e33f8-6f5a-4887-a9bd-0719cccd5efa
To: /content/_loader.py
100% 1.90k/1.90k [00:00<00:00, 8.48MB/s]


In [7]:
pipe = pipeline(

    task="depth-estimation",

    model="depth-anything/Depth-Anything-V2-Small-hf",

    device=device

)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


## продолжение

In [8]:
import os

def predict_depth(image_path, save_path=None):

    """

    Предсказать depth для одного изображения.

    Возвращает np.array (H, W) с относительной глубиной 0–1.

    """

    image = Image.open(image_path).convert('RGB')

    result = pipe(image)

    depth = np.array(result['depth'])  # (H, W), относительная



    # Нормализуем в 0–1

    depth_normalized = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)



    if save_path:

        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        np.save(save_path, depth_normalized.astype(np.float32))



    return depth_normalized

In [10]:
from tqdm.notebook import tqdm
from _loader import VKITTI2Loader

VKITTI_RGB = '/content/vkitti2'

DEPTH_PRED_DIR = '/content/depth_pred_vkitti'

loader = VKITTI2Loader(VKITTI_RGB)


In [11]:
for scene in loader.SCENES:

    frames = loader.list_frames(scene, 'clone')

    print(f'n{scene}: {len(frames)} кадров')



    for frame_id in tqdm(frames, desc=scene):

        rgb_path = f'{VKITTI_RGB}/{scene}/clone/frames/rgb/Camera_0/rgb_{frame_id}.jpg'

        save_path = f'{DEPTH_PRED_DIR}/{scene}/{frame_id}.npy'

        if not os.path.exists(save_path):

            predict_depth(rgb_path, save_path)

print('n✅ Инференс на VKITTI2 завершён')

nScene01: 447 кадров


Scene01:   0%|          | 0/447 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


nScene02: 233 кадров


Scene02:   0%|          | 0/233 [00:00<?, ?it/s]

...

nScene18: 339 кадров


Scene18:   0%|          | 0/339 [00:00<?, ?it/s]

nScene20: 837 кадров


Scene20:   0%|          | 0/837 [00:00<?, ?it/s]

n✅ Инференс на VKITTI2 завершён


In [12]:
!zip -r depth_pred_vkitti.zip /content/depth_pred_vkitti

  adding: content/depth_pred_vkitti/ (stored 0%)
  adding: content/depth_pred_vkitti/Scene01/ (stored 0%)
  adding: content/depth_pred_vkitti/Scene01/00013.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00124.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00288.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00144.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00162.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00001.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00265.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00206.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00291.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00321.npy (deflated 96%)
  adding: content/depth_pred_vkitti/Scene01/00382.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00197.npy (deflated 97%)
  adding: content/depth_pred_vkitti/Scene01/00379.npy (deflated 97

In [13]:
!ls

depth_pred_vkitti      _loader.py   sample_data  vkitti_2.0.3_depth.tar
depth_pred_vkitti.zip  __pycache__  vkitti2	 vkitti_2.0.3_rgb.tar


In [14]:
from google.colab import files
files.download('depth_pred_vkitti.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>